In [1]:
import pandas as pd 
import numpy as np
import os
from pathlib import Path
from typing import Optional,Tuple, List

## 1.0 Import dataset

In [2]:
# import train data 
df = pd.read_csv("/home/sjoon/projects/brain_connectivity_classifier/data/raw/PIOP2_restingstate.csv")
df.shape

(224, 26797)

## 2. Data Preparation

In [3]:
# Extract connection columns
connection_columns = [col for col in df.columns if '~' in str(col)]

# Extract actual regions from connection columns
def extract_regions(connection_columns):
    unique_regions = []
    seen = set()
    
    for col in connection_columns:
        region_a, region_b = col.split('~', 1)
        for region in [region_a, region_b]:
            if region not in seen:
                seen.add(region)
                unique_regions.append(region)
    
    region_to_idx = {region: idx for idx, region in enumerate(unique_regions)}
    n_regions = len(unique_regions)
    
    return unique_regions, region_to_idx, n_regions

# Extract from your actual data
region_list, region_to_idx, n_regions = extract_regions(connection_columns)

# Print results
print(f"Found {n_regions} regions")
print(f"Sample regions: {region_list[:3]}")
print(f"Connection columns: {len(connection_columns)}")

Found 232 regions
Sample regions: ['LH_VisCent_ExStr_2', 'LH_VisCent_ExStr_1', 'LH_VisCent_Striate_1']
Connection columns: 26796


In [4]:
def reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions):
    n_subjects = df.shape[0]
    matrices = np.zeros((n_subjects, n_regions, n_regions))
    
    values = df[connection_columns].values
    
    for subj_idx in range(n_subjects):
        matrix = matrices[subj_idx]
        for col_idx, col in enumerate(connection_columns):
            region_a, region_b = col.split('~', 1)
            idx_a = region_to_idx[region_a]
            idx_b = region_to_idx[region_b]
            
            # Place value symmetrically
            value = values[subj_idx, col_idx]
            matrix[idx_a, idx_b] = value
            matrix[idx_b, idx_a] = value
        
        # Self-correlations
        np.fill_diagonal(matrix, 1.0)
    
    return matrices 

df_mat = reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions)
df_mat.shape

(224, 232, 232)

In [5]:
df_mat[0].shape

(232, 232)

## 3. Diagonal Imputation

In [6]:
def reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions):
    n_subjects = df.shape[0]
    matrices = np.zeros((n_subjects, n_regions, n_regions))
    
    values = df[connection_columns].values
    
    for subj_idx in range(n_subjects):
        matrix = matrices[subj_idx]
        for col_idx, col in enumerate(connection_columns):
            region_a, region_b = col.split('~', 1)
            idx_a = region_to_idx[region_a]
            idx_b = region_to_idx[region_b]
            
            # Place value symmetrically
            value = values[subj_idx, col_idx]
            matrix[idx_a, idx_b] = value
            matrix[idx_b, idx_a] = value
        
        # Self-correlations
        np.fill_diagonal(matrix, 1.0)
    
    return matrices 

df_mat = reconstruct_matrices_from_dataframe(df, connection_columns, region_to_idx, n_regions)
df_mat.shape

(224, 232, 232)

In [7]:
# Extract diagonal and flatten
diagonal_values = np.diagonal(df_mat[0])

# first 30 diagonal values of subject 1
diagonal_values[0:30]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [8]:
# first 5 rows and columns of subject 0
df_mat[0][0:5, 0:5]

array([[1.        , 0.44923772, 0.52421013, 0.27670956, 0.32092815],
       [0.44923772, 1.        , 0.73007062, 0.49777039, 0.53532657],
       [0.52421013, 0.73007062, 1.        , 0.56103755, 0.32186132],
       [0.27670956, 0.49777039, 0.56103755, 1.        , 0.59854227],
       [0.32092815, 0.53532657, 0.32186132, 0.59854227, 1.        ]])

### 3.1 Impute diagonal with precision of off diagonal values 

In [9]:
def impute_diagonal_precision(
        matrices: np.ndarray,
        regularization: str = 'tikhonov',
        alpha: float = 0.1,
        normalize: bool = True,
) -> np.ndarray:
    """
    Impute diagonal using precision matrix (inverse covariance).

    NEUROSCIENCE BASIS:
    - Diagonal of precision matrix ≈ strength of anatomical self-connections
    - Stronger in sensory/motor regions
    - Captures direct dependencies (partial correlations)

    Args:
        matrices: (n_subjects, n_regions, n_regions) correlation matrices with diag = 1.0
        region_list: Optional list of region names (unused here but kept for API consistency)
        regularization: 'tikhonov' (recommended), 'none'
        alpha: Regularization strength (Tikhonov); typical [0.01–0.5]
        normalize: Scale precision diagonal to reasonable range

    Returns:
        matrices_imputed: Same shape, with diagonal replaced by precision diagonal
    """
    results = matrices.copy()
    n_subjects, n_regions, _ = matrices.shape

    for s in range(n_subjects):
        corr = matrices[s].copy()

        # Ensure valid correlation matrix 
        np.fill_diagonal(corr, 1.0) 
        corr = (corr + corr.T) / 2.0  # enforce perfect symmetry

        # Apply regularization
        if regularization == 'tikhonov':  # ridge regularization (L2)
            regularized = corr + alpha * np.eye(n_regions)
        elif regularization == 'none':
            regularized = corr
        else: 
            raise ValueError(f"Invalid regularization: {regularization}")
        
        # Invert to get precision matrix
        precision = np.linalg.inv(regularized)
        precision_diagonal = np.diag(precision)

        # Normalise to avoid extreme values 
        if normalize:
            max_abs = np.max(np.abs(precision_diagonal)) 
            
            # Scale to [-1,1] based on max_abs value
            if max_abs > 1.0:
                precision_diagonal = precision_diagonal / max_abs
        
        # Replace diagonal
        np.fill_diagonal(results[s], precision_diagonal)

    return results


# CORRECT: Pass the entire 3D array at once
df_mat_imputed = impute_diagonal_precision(df_mat, alpha=0.9)

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_mat_imputed[0])
diagonal_values_2 = np.diagonal(df_mat_imputed[1])

# First 30 diagonal values of subject 0
print(diagonal_values_1[0:30])
print('')
print(diagonal_values_2[0:30])

# first 5 
print(f'\n 5 rows x 5 col of subject 0 \n {df_mat_imputed[0][0:5, 0:5]} ')


[0.91555845 0.93335636 0.96800688 0.93984715 0.96131952 0.9654493
 0.93515069 0.93752823 0.91598681 0.97874536 0.97361805 0.98824623
 0.91421987 0.944027   0.94900901 0.95293501 0.92483702 0.96003846
 0.87350448 0.89907837 0.94849569 0.93689296 0.95014195 0.98541805
 0.90693792 0.9332309  0.93839578 0.95990039 0.96462422 0.91904367]

[0.89734091 0.91543967 0.85362915 0.93918615 0.97107623 0.97060867
 0.95261034 0.89213593 0.91320136 0.93473623 0.92004923 0.95574744
 0.9471876  0.90025334 0.93113372 0.88246476 0.92013498 0.9321457
 0.93305547 0.91378446 0.91645411 0.91261681 0.95288515 0.9610299
 0.94237983 0.9703547  0.91645594 0.90117261 0.90349277 0.9605284 ]

 5 rows x 5 col of subject 0 
 [[0.91555845 0.44923772 0.52421013 0.27670956 0.32092815]
 [0.44923772 0.93335636 0.73007062 0.49777039 0.53532657]
 [0.52421013 0.73007062 0.96800688 0.56103755 0.32186132]
 [0.27670956 0.49777039 0.56103755 0.93984715 0.59854227]
 [0.32092815 0.53532657 0.32186132 0.59854227 0.96131952]] 


## 4. Preprocessing

### Fisher-z transformation

In [10]:
def fisher_z_transform(matrix, clip_value=0.999999):
    # Clip values to avoid infinities at r = ±1
    clipped_matrix = np.clip(matrix, -clip_value, clip_value)
    
    # Apply transformation to the CLIPPED matrix
    return np.arctanh(clipped_matrix)

# Apply fisher z transformation
df_mat_fz = np.array([fisher_z_transform(df_mat_imputed[i]) for i in range(df_mat_imputed.shape[0])])

# Extract diagonal from first subject
diagonal_values_1 = np.diagonal(df_mat_fz[0])
diagonal_values_2 = np.diagonal(df_mat_fz[1])

# First 30 diagonal values of subject 0
print(f'First 30 diagonal values of subject 0 \n {diagonal_values_1[0:30]}')
print('')
print(f'First 30 diagonal values of subject 1 \n {diagonal_values_1[0:30]}')


First 30 diagonal values of subject 0 
 [1.56085247 1.68382663 2.05962791 1.73673783 1.96301887 2.02052418
 1.69793711 1.71722675 1.56350713 2.26682147 2.15747191 2.565417
 1.55263902 1.77382368 1.82171369 1.8627796  1.6214689  1.94640097
 1.34767957 1.46738983 1.81657371 1.71200407 1.83323878 2.45690001
 1.50999365 1.68285374 1.72444283 1.94464117 2.00851441 1.58283619]

First 30 diagonal values of subject 1 
 [1.56085247 1.68382663 2.05962791 1.73673783 1.96301887 2.02052418
 1.69793711 1.71722675 1.56350713 2.26682147 2.15747191 2.565417
 1.55263902 1.77382368 1.82171369 1.8627796  1.6214689  1.94640097
 1.34767957 1.46738983 1.81657371 1.71200407 1.83323878 2.45690001
 1.50999365 1.68285374 1.72444283 1.94464117 2.00851441 1.58283619]


In [11]:
print(f'5 rows x 5 col of subject 0 \n {df_mat_fz[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_mat_fz[1][0:5, 0:5]} ')

5 rows x 5 col of subject 0 
 [[1.56085247 0.48374485 0.58212765 0.28411527 0.33268149]
 [0.48374485 1.68382663 0.92887857 0.54633774 0.59758176]
 [0.58212765 0.92887857 2.05962791 0.63434606 0.33372215]
 [0.28411527 0.54633774 0.63434606 1.73673783 0.69087258]
 [0.33268149 0.59758176 0.33372215 0.69087258 1.96301887]] 
 
 5 rows x 5 col of subject 1 
 [[1.4583975  0.41619344 0.20542718 0.54997872 0.5313618 ]
 [0.41619344 1.56011859 0.41847708 0.42191381 0.86791993]
 [0.20542718 0.41847708 1.26937861 0.58216001 0.30742853]
 [0.54997872 0.42191381 0.58216001 1.73110301 0.74325942]
 [0.5313618  0.86791993 0.30742853 0.74325942 2.11083562]] 


### Standard Scaler

In [12]:
from sklearn.preprocessing import StandardScaler

# Apply scaling to EACH subject's matrix (not the whole 3D array)
df_mat_fz_scaled = np.array([StandardScaler().fit_transform(df_mat_fz[i]) for i in range(df_mat_fz.shape[0])])

print(f'5 rows x 5 col of subject 0 \n {df_mat_fz_scaled[0][0:5, 0:5]} ')
print(f' \n 5 rows x 5 col of subject 1 \n {df_mat_fz_scaled[1][0:5, 0:5]} ')

5 rows x 5 col of subject 0 
 [[6.60414626 1.60979561 1.84656221 0.99936529 1.01850961]
 [1.98054056 6.04976423 2.99213806 1.93908262 1.91013835]
 [2.40285979 3.25666648 6.72784309 2.25447491 1.02201238]
 [1.12360812 1.84137187 2.0190784  6.20507712 2.22414623]
 [1.33208408 2.03096051 1.02589433 2.45704701 6.50606775]] 
 
 5 rows x 5 col of subject 1 
 [[6.19355458 1.22683091 1.08699899 2.06040017 1.38762549]
 [1.77752836 4.68047978 2.27782006 1.60037321 2.29873022]
 [0.88446973 1.23372551 7.03384869 2.17599984 0.78141026]
 [2.34440327 1.2441014  3.19270936 6.30316257 1.96125872]
 [2.26551966 2.59064801 1.65712532 2.75469127 5.66345766]] 


## 5. Modelling (StandardScaler)

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Prepare indices
X = df_mat.reshape(-1, 232)
y = np.tile(np.arange(232), 224)
groups = np.repeat(np.arange(224), 232)

gkf = GroupKFold(n_splits=3)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    # Get unique subjects in train/val
    train_subjects = np.unique(groups[train_idx])
    val_subjects = np.unique(groups[val_idx])

    print(f"Fold {fold+1}/{gkf.get_n_splits()}")
    print(f"Train subjects: {train_subjects}")
    print(f"Val subjects: {val_subjects}")
    
    # Extract 3D matrices for these subjects
    train_mat_3d = df_mat[train_subjects]  # (n_train_subjects, 232, 232)
    val_mat_3d = df_mat[val_subjects]      # (n_val_subjects, 232, 232)
    
    # Apply diagonal imputation on 3D matrices
    train_mat_imputed = impute_diagonal_precision(train_mat_3d, alpha=0.9)
    val_mat_imputed = impute_diagonal_precision(val_mat_3d)
    
    # Reshape back to 2D
    X_train_fold = train_mat_imputed.reshape(-1, 232)
    X_val_fold = val_mat_imputed.reshape(-1, 232)
    y_train_fold = y[train_idx]
    y_val_fold = y[val_idx]
    
    # Fisher Z transform
    X_train_fz = fisher_z_transform(X_train_fold)
    X_val_fz = fisher_z_transform(X_val_fold)
    
    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_fz)
    X_val_scaled = scaler.transform(X_val_fz)
    
    # Train
    model = LogisticRegression(
        C=0.0343304473310619,
        max_iter=1000,
        solver='saga',
        multi_class='multinomial'
    )
    model.fit(X_train_scaled, y_train_fold)
    
    score = accuracy_score(y_val_fold, model.predict(X_val_scaled))
    fold_scores.append(score)
    print(f"Fold {fold + 1} Accuracy: {score:.4f}")

print(f"\nMean CV Accuracy: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

Fold 1/3
Train subjects: [  0   2   3   5   6   8   9  11  12  14  15  17  18  20  21  23  24  26
  27  29  30  32  33  35  36  38  39  41  42  44  45  47  48  50  51  53
  54  56  57  59  60  62  63  65  66  68  69  71  72  74  75  77  78  80
  81  83  84  86  87  89  90  92  93  95  96  98  99 101 102 104 105 107
 108 110 111 113 114 116 117 119 120 122 123 125 126 128 129 131 132 134
 135 137 138 140 141 143 144 146 147 149 150 152 153 155 156 158 159 161
 162 164 165 167 168 170 171 173 174 176 177 179 180 182 183 185 186 188
 189 191 192 194 195 197 198 200 201 203 204 206 207 209 210 212 213 215
 216 218 219 221 222]
Val subjects: [  1   4   7  10  13  16  19  22  25  28  31  34  37  40  43  46  49  52
  55  58  61  64  67  70  73  76  79  82  85  88  91  94  97 100 103 106
 109 112 115 118 121 124 127 130 133 136 139 142 145 148 151 154 157 160
 163 166 169 172 175 178 181 184 187 190 193 196 199 202 205 208 211 214
 217 220 223]


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 1 Accuracy: 0.9899
Fold 2/3
Train subjects: [  1   2   4   5   7   8  10  11  13  14  16  17  19  20  22  23  25  26
  28  29  31  32  34  35  37  38  40  41  43  44  46  47  49  50  52  53
  55  56  58  59  61  62  64  65  67  68  70  71  73  74  76  77  79  80
  82  83  85  86  88  89  91  92  94  95  97  98 100 101 103 104 106 107
 109 110 112 113 115 116 118 119 121 122 124 125 127 128 130 131 133 134
 136 137 139 140 142 143 145 146 148 149 151 152 154 155 157 158 160 161
 163 164 166 167 169 170 172 173 175 176 178 179 181 182 184 185 187 188
 190 191 193 194 196 197 199 200 202 203 205 206 208 209 211 212 214 215
 217 218 220 221 223]
Val subjects: [  0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51
  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 102 105
 108 111 114 117 120 123 126 129 132 135 138 141 144 147 150 153 156 159
 162 165 168 171 174 177 180 183 186 189 192 195 198 201 204 207 210 213
 216 219 222]


/home/sjoon/projects/brain_connectivity_classifier/masterthesis_venv2/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Fold 2 Accuracy: 0.9907
Fold 3/3
Train subjects: [  0   1   3   4   6   7   9  10  12  13  15  16  18  19  21  22  24  25
  27  28  30  31  33  34  36  37  39  40  42  43  45  46  48  49  51  52
  54  55  57  58  60  61  63  64  66  67  69  70  72  73  75  76  78  79
  81  82  84  85  87  88  90  91  93  94  96  97  99 100 102 103 105 106
 108 109 111 112 114 115 117 118 120 121 123 124 126 127 129 130 132 133
 135 136 138 139 141 142 144 145 147 148 150 151 153 154 156 157 159 160
 162 163 165 166 168 169 171 172 174 175 177 178 180 181 183 184 186 187
 189 190 192 193 195 196 198 199 201 202 204 205 207 208 210 211 213 214
 216 217 219 220 222 223]
Val subjects: [  2   5   8  11  14  17  20  23  26  29  32  35  38  41  44  47  50  53
  56  59  62  65  68  71  74  77  80  83  86  89  92  95  98 101 104 107
 110 113 116 119 122 125 128 131 134 137 140 143 146 149 152 155 158 161
 164 167 170 173 176 179 182 185 188 191 194 197 200 203 206 209 212 215
 218 221]
